In [1]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from api_keys import GEMINI_1

os.environ["GOOGLE_API_KEY"] = GEMINI_1

MODEL = "gemini-3-flash-preview"
llm = ChatGoogleGenerativeAI(
    model=f"models/{MODEL}",
    temperature=1.0,
    max_tokens=4096,
    max_retries=6,  # Ile razy ma próbować ponownie przy błędzie limitu
)

def ask_llm(question):
    try:
        message = HumanMessage(content=question)
        response = llm.invoke([message])
        return response.content
    except Exception as e:
        return f"Błąd: {e}"

In [2]:
import json
import random
with open('articles_topics_processed.json', 'r', encoding='utf-8') as fp:
    topics = json.load(fp)
    random.shuffle(topics)

In [3]:
try:
    generated = os.listdir(MODEL)
    generated = [int(x[:-5]) for x in generated]
except FileNotFoundError:
    generated = []

In [4]:
os.makedirs(MODEL, exist_ok=True)

In [5]:
import json
for a in topics:
    if a['ID'] in generated:
        continue
    pytanie = f"""
    Jesteś autorem artykułów dla popularnych polskich mediów.
    Napisz artykuł na temat podany poniżej. Weź pod uwagę, wszystkie informacje,
    które zaraz Ci podam. Czyli: 
    - Zdarzenie, którego to dotyczy,
    - Temat artykułu,
    - Długość i styl artykułu
    
    Napisz artykuł z podanymi cechami:
    Zdarzenie: {a['Zdarzenie']},
    Temat: {a['Temat']},
    Długość i styl artykułu: {a['Długość']}

    Napisz to w formie
    
    Tytuł: 
    Treść:
    
    Nie podawaj żadnej dodatkowej treści typu: Oto artykuł czy jakiś znaków specjalnych"""

    t = ask_llm(pytanie)
    t = t[0]['text']
    frag = t.split('Treść:')
    a['Wygenerowany tytuł'] = frag[0].split('Tytuł: ')[-1]
    a['Wygenerowany tekst'] = frag[1]
    a['Model'] = f"models/{MODEL}"
    with open(f"{MODEL}/{a['ID']}.json", "w", encoding='utf-8') as f:
        json.dump(a, f, ensure_ascii=False)
    print(f'{a["ID"]} / {len(topics)}')

282 / 2761
1284 / 2761
334 / 2761
738 / 2761
201 / 2761
2696 / 2761
2282 / 2761
1667 / 2761
1670 / 2761
2698 / 2761
781 / 2761
1282 / 2761
2692 / 2761
845 / 2761
951 / 2761
2216 / 2761
451 / 2761
2098 / 2761
871 / 2761


TypeError: string indices must be integers, not 'str'